In [ ]:
import numpy as np
import pandas as pd
import json
from datetime import datetime, timedelta
import pycountry

from emu_renewal.constants import DATA_PATH, ANALYSIS_TYPES, DATE_FORMAT, RUN_DATA_DELAY
from emu_renewal.inputs import get_cont_of_country, get_country_pop
from emu_renewal.renew import MultiStrainModel
from emu_renewal.run import run_single_country, find_run_start_time, find_run_end_time, get_country_vars, get_alpha_info, get_delta_info, get_mobility_provider

In [ ]:
countries = json.load(open(DATA_PATH / f"config/included.json", "r"))
country = pycountry.countries.lookup("France")
mob_source = "no_mob"
iso3 = country.alpha_3
continent = get_cont_of_country(iso3)
pop = get_country_pop(iso3)
data_start = find_run_start_time(pop, iso3)
end_time = find_run_end_time(iso3, mob_source)
run_start = data_start - timedelta(RUN_DATA_DELAY)
start_str = run_start.strftime(DATE_FORMAT)
end_str = data_start.strftime(DATE_FORMAT)
var_data = get_country_vars(iso3)
delta_var, delta_targ, delta_seed = get_delta_info(iso3, var_data, continent, end_time)
alpha_var, alpha_targ, alpha_seed = get_alpha_info(iso3, var_data, continent, end_time, delta_targ)
start_var = "eu"
var_names = [start_var] + alpha_var
seed_times = [] + alpha_seed
mob_provider = get_mobility_provider(iso3, mob_source)
model = MultiStrainModel(pop, run_start, end_time, var_names, seed_times, mob_provider, False)

In [ ]:
results = model.renewal_func(
    np.random.normal(size=62), 
    4.0, 
    2.0, 
    0.4,
    0.008,
    0.8,
    8.0,
    3.0,
    25.0,
    3.0,
    15.0,
    2.0,
    7.0,  # Unused
    1.0,  # Unused
    0.035,
    10.0,  # Unused
    1.0,  # Unused
    7.0,  # Unused
    1.0,  # Unused
    0.02,  # Unused
    0.5,
    np.array([-13.0, -14.0]),
    np.array([1.5]),
    np.array([50.0]),
    0.0,  # Unused
    0.0,  # Unused
)

In [ ]:
pd.Series(results["weekly_deaths"]).plot()